In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

In [3]:
df = pd.read_csv("Dataset/Train/ML/final_dataset.csv")

X = df.drop(columns=["Stress_Level","Subject_ID","Task_Type"])
y = df["Stress_Level"]

groups = df["Subject_ID"]

print("X Shape :", X.shape)
print("y Shape :", y.shape)
print("Unique Subjects :", groups.nunique())

X Shape : (11036, 96)
y Shape : (11036,)
Unique Subjects : 15


In [4]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups),start=1):

    train_subjects = groups.iloc[train_idx].unique()
    test_subjects = groups.iloc[test_idx].unique()

    print(f"\nFold {fold}")

    print(
        "Train Subjects:",
        sorted(train_subjects)
    )

    print(
        "Test Subjects:",
        sorted(test_subjects)
    )

    print(
        "Overlap:",
        set(train_subjects).intersection(
            set(test_subjects)
        )
    )




Fold 1
Train Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(7), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14)]
Test Subjects: [np.int64(6), np.int64(9), np.int64(15)]
Overlap: set()

Fold 2
Train Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(7), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(14), np.int64(15)]
Test Subjects: [np.int64(5), np.int64(8), np.int64(13)]
Overlap: set()

Fold 3
Train Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(11), np.int64(12), np.int64(13), np.int64(15)]
Test Subjects: [np.int64(4), np.int64(10), np.int64(14)]
Overlap: set()

Fold 4
Train Subjects: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(8), np.int64(9), np.int64(10), np.int64(13), np.int64(14), np.int64(15)]
Test Subjects: [np.int64(7), np.int64(11), n

In [5]:
import pandas as pd

pd.crosstab(
    df["Task_Type"],
    df["Stress_Level"]
)

Stress_Level,highlevel,lowlevel,midlevel,natural
Task_Type,,,,
Arithmetic,1780,1121,1240,1706
Stroop,1522,1001,960,1706


In [6]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import numpy as np
import pandas as pd


def evaluate_model_cv(model, X, y, groups, n_splits=5):

    gkf = GroupKFold(n_splits=n_splits)

    accuracy_scores = []; precision_scores = []
    recall_scores = []; f1_scores = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), start=1):

        X_train = X.iloc[train_idx]; y_train = y.iloc[train_idx]
        X_test = X.iloc[test_idx]; y_test = y.iloc[test_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(
            y_test,
            y_pred,
            average="weighted"
        )
        rec = recall_score(
            y_test,
            y_pred,
            average="weighted"
        )
        f1 = f1_score(
            y_test,
            y_pred,
            average="weighted"
        )

        accuracy_scores.append(acc)
        precision_scores.append(prec)
        recall_scores.append(rec)
        f1_scores.append(f1)

        print(f"\nFold {fold}")
        print(f"Precision : {prec:.4f}",end="  ")
        print(f"Recall : {rec:.4f}",end="  ")
        print(f"F1 Score : {f1:.4f}",end="  ")
        print(f"Accuracy : {acc:.4f}")

    results = {
        "Model": model.__class__.__name__,
        "Mean Accuracy": np.mean(accuracy_scores),
        "Mean Precision": np.mean(precision_scores),
        "Mean Recall": np.mean(recall_scores),
        "Mean F1": np.mean(f1_scores),
        "Std Accuracy": np.std(accuracy_scores)
    }

    print("\n===== FINAL RESULTS =====")

    for key, value in results.items():

        if key == "Model":
            print(f"{key}: {value}")
        else:
            print(f"{key}: {value:.4f}")

    return results

### Random Forest Baseline

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_result = evaluate_model_cv(
    rf,
    X,
    y,
    groups
)


Fold 1
Precision : 0.4993  Recall : 0.5824  F1 Score : 0.5279  Accuracy : 0.5824

Fold 2
Precision : 0.5593  Recall : 0.4679  F1 Score : 0.5005  Accuracy : 0.4679

Fold 3
Precision : 0.3226  Recall : 0.2816  F1 Score : 0.2497  Accuracy : 0.2816

Fold 4
Precision : 0.3079  Recall : 0.2445  F1 Score : 0.2418  Accuracy : 0.2445

Fold 5
Precision : 0.3159  Recall : 0.2953  F1 Score : 0.2950  Accuracy : 0.2953

===== FINAL RESULTS =====
Model: RandomForestClassifier
Mean Accuracy: 0.3743
Mean Precision: 0.4010
Mean Recall: 0.3743
Mean F1: 0.3630
Std Accuracy: 0.1294


In [8]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

print(feature_importance.head(20))

      Feature  Importance
34    F7_beta    0.048399
11  FP1_gamma    0.035017
71   F4_gamma    0.034049
82    F8_beta    0.031969
47   F3_gamma    0.030508
83   F8_gamma    0.030314
95   C4_gamma    0.029023
59   FZ_gamma    0.027671
10   FP1_beta    0.026040
35   F7_gamma    0.025991
58    FZ_beta    0.021279
23  FP2_gamma    0.021103
49     FZ_std    0.021093
4   FP1_range    0.021086
52   FZ_range    0.021024
22   FP2_beta    0.020508
89     C4_var    0.020218
85     C4_std    0.020120
70    F4_beta    0.018176
53     FZ_var    0.017737


In [9]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(
    n_neighbors=5
)

knn_result = evaluate_model_cv(
    knn,
    X,
    y,
    groups
)


Fold 1
Precision : 0.4976  Recall : 0.5311  F1 Score : 0.4982  Accuracy : 0.5311

Fold 2
Precision : 0.4929  Recall : 0.4683  F1 Score : 0.4773  Accuracy : 0.4683

Fold 3
Precision : 0.3551  Recall : 0.3509  F1 Score : 0.3509  Accuracy : 0.3509

Fold 4
Precision : 0.3382  Recall : 0.3125  F1 Score : 0.3148  Accuracy : 0.3125

Fold 5
Precision : 0.3403  Recall : 0.3228  F1 Score : 0.3220  Accuracy : 0.3228

===== FINAL RESULTS =====
Model: KNeighborsClassifier
Mean Accuracy: 0.3971
Mean Precision: 0.4048
Mean Recall: 0.3971
Mean F1: 0.3926
Std Accuracy: 0.0870


In [10]:
from sklearn.svm import SVC

svm = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale"
)

svm_result = evaluate_model_cv(
    svm,
    X,
    y,
    groups
)


Fold 1
Precision : 0.4501  Recall : 0.5057  F1 Score : 0.4516  Accuracy : 0.5057

Fold 2
Precision : 0.5090  Recall : 0.5142  F1 Score : 0.5025  Accuracy : 0.5142

Fold 3
Precision : 0.3932  Recall : 0.3083  F1 Score : 0.3259  Accuracy : 0.3083

Fold 4
Precision : 0.2813  Recall : 0.2837  F1 Score : 0.2719  Accuracy : 0.2837

Fold 5
Precision : 0.3928  Recall : 0.3922  F1 Score : 0.3661  Accuracy : 0.3922

===== FINAL RESULTS =====
Model: SVC
Mean Accuracy: 0.4008
Mean Precision: 0.4053
Mean Recall: 0.4008
Mean F1: 0.3836
Std Accuracy: 0.0961


In [11]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=5000
)

lr_result = evaluate_model_cv(
    lr,
    X,
    y,
    groups
)


Fold 1
Precision : 0.4533  Recall : 0.5018  F1 Score : 0.4471  Accuracy : 0.5018

Fold 2
Precision : 0.5275  Recall : 0.5326  F1 Score : 0.4843  Accuracy : 0.5326

Fold 3
Precision : 0.3920  Recall : 0.3123  F1 Score : 0.3216  Accuracy : 0.3123

Fold 4
Precision : 0.2170  Recall : 0.2212  F1 Score : 0.2046  Accuracy : 0.2212

Fold 5
Precision : 0.4273  Recall : 0.4024  F1 Score : 0.3500  Accuracy : 0.4024

===== FINAL RESULTS =====
Model: LogisticRegression
Mean Accuracy: 0.3941
Mean Precision: 0.4034
Mean Recall: 0.3941
Mean F1: 0.3615
Std Accuracy: 0.1161


In [14]:
comparison_df = pd.DataFrame([
    rf_result,
    svm_result,
])

comparison_df.sort_values(
    by="Mean Accuracy",
    ascending=False
)

,Model,Mean Accuracy,Mean Precision,Mean Recall,Mean F1,Std Accuracy
1,SVC,0.400815,0.405274,0.400815,0.383587,0.096141
0,RandomForestClassifier,0.374335,0.400982,0.374335,0.362957,0.129412
